Monitoraggio della reputazione online di un’azienda
MachineInnovators Inc. è leader nello sviluppo di applicazioni di machine learning scalabili e pronte per la produzione. Il focus principale del progetto è integrare metodologie MLOps per facilitare lo sviluppo, l'implementazione, il monitoraggio continuo e il retraining dei modelli di analisi del sentiment. L'obiettivo è abilitare l'azienda a migliorare e monitorare la reputazione sui social media attraverso l'analisi automatica dei sentiment.

Le aziende si trovano spesso a fronteggiare la sfida di gestire e migliorare la propria reputazione sui social media in modo efficace e tempestivo. Monitorare manualmente i sentiment degli utenti può essere inefficiente e soggetto a errori umani, mentre la necessità di rispondere rapidamente ai cambiamenti nel sentiment degli utenti è cruciale per mantenere un'immagine positiva dell'azienda.

Benefici della Soluzione

Automazione dell'Analisi del sentiment: Implementando un modello di analisi del sentiment, MLOps Innovators Inc. automatizzerà l'elaborazione dei dati dai social media per identificare sentiment positivi, neutrali e negativi. Ciò permetterà una risposta rapida e mirata ai feedback degli utenti.
Monitoraggio Continuo della Reputazione: Utilizzando metodologie MLOps, l'azienda implementerà un sistema di monitoraggio continuo per valutare l'andamento del sentiment degli utenti nel tempo. Questo consentirà di rilevare rapidamente cambiamenti nella percezione dell'azienda e di intervenire prontamente se necessario.
Retraining del Modello: Introdurre un sistema di retraining automatico per il modello di analisi del sentiment assicurerà che l'algoritmo si adatti dinamicamente ai nuovi dati e alle variazioni nel linguaggio e nei comportamenti degli utenti sui social media. Mantenere alta l'accuratezza predittiva del modello è essenziale per una valutazione corretta del sentiment.
Dettagli del Progetto

Fase 1: Implementazione del Modello di Analisi del sentiment
Modello: Utilizzare un modello pre-addestrato per un’analisi del sentiment in grado di classificare testi dai social media in sentiment positivo, neutro o negativo. Servirsi di questo modello: https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest
Dataset: Utilizzare dataset pubblici contenenti testi e le rispettive etichette di sentiment.
Fase 2: Creazione della Pipeline CI/CD
Pipeline CI/CD: Sviluppare una pipeline automatizzata per il training del modello, i test di integrazione e il deploy dell'applicazione su HuggingFace.
Fase 3: Deploy e Monitoraggio Continuo
Deploy su HuggingFace (facoltativo): Implementare il modello di analisi del sentiment, inclusi dati e applicazione, su HuggingFace per facilitare l'integrazione e la scalabilità.
Sistema di Monitoraggio: Configurare un sistema di monitoraggio per valutare continuamente le performance del modello e il sentiment rilevato.
Consegna
Codice Sorgente: Repository pubblica su GitHub con codice ben documentato per la pipeline CI/CD e l'implementazione del modello. La consegna vera e propria dovrà avvenire mediante un notebook google colab con al suo interno il link al repository GitHub.
Documentazione: Descrizione delle scelte progettuali, delle implementazioni e dei risultati ottenuti durante il progetto.
Motivazione del Progetto

L'implementazione di un modello per l'analisi del sentiment consente a MLOps Innovators Inc. di migliorare significativamente la gestione della reputazione sui social media. Automatizzando l'analisi del sentiment, l'azienda potrà rispondere più rapidamente alle esigenze degli utenti, migliorando la soddisfazione e rafforzando l'immagine dell'azienda sul mercato. Con questo progetto, MLOps Innovators Inc. promuove l'innovazione nel campo delle tecnologie AI, offrendo soluzioni avanzate e scalabili per le sfide moderne di gestione della reputazione aziendale.

Modalità di consegna:
Link pubblico a notebook di Google Colab


## 1. Introduzione e obiettivi

MachineInnovators Inc. sviluppa applicazioni di machine learning scalabili e pronte per la produzione. Questo progetto implementa un sistema **MLOps** completo per l'analisi automatica del sentiment sui social media, con l'obiettivo di:

- **Automatizzare** la classificazione dei post/commenti social in *positivo*, *neutro*, *negativo*;
- **Monitorare in continuo** l'andamento della reputazione aziendale nel tempo;
- **Ri-addestrare automaticamente** il modello quando le performance degradano o cambiano i pattern linguistici degli utenti.

**Modello utilizzato:** [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest) — RoBERTa fine-tuned su ~124M tweet, scelto perché addestrato specificamente su linguaggio da social media (informale, hashtag, emoji), a differenza di modelli generici addestrati su recensioni o news.

**Dataset pubblico:** [`tweet_eval`](https://huggingface.co/datasets/tweet_eval) (subset `sentiment`), con le stesse 3 classi del modello.


In [ ]:
# Installazione delle dipendenze necessarie (Colab)
!pip install -q transformers datasets scikit-learn pandas matplotlib evaluate gradio huggingface_hub


## 2. Fase 1 — Modello di Sentiment Analysis

Carichiamo il modello pre-addestrato e testiamo l'inferenza su alcuni esempi tipici di post/commenti social riferiti a un'azienda fittizia.


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
sentiment_pipe = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

print("Modello caricato:", MODEL_NAME)
print("Etichette:", model.config.id2label)


In [ ]:
# Esempi di post/commenti social riferiti a MachineInnovators Inc.
esempi = [
    "Il nuovo prodotto di MachineInnovators è fantastico, lo consiglio a tutti!",
    "Il servizio clienti non ha risolto il mio problema, molto deluso.",
    "Ho ricevuto l'ordine in tempo, niente di particolare da segnalare.",
    "Team di supporto gentilissimo, hanno risolto tutto in 5 minuti!",
    "App che si blocca in continuazione, esperienza pessima.",
]

risultati = sentiment_pipe(esempi)
for testo, r in zip(esempi, risultati):
    print(f"[{r['label']:>8} | {r['score']:.3f}]  {testo}")


> **Osservazione importante:** il modello classifica la frase chiaramente negativa *"App che si blocca in continuazione, esperienza pessima"* come `neutral`. Questo perché `cardiffnlp/twitter-roberta-base-sentiment-latest` è addestrato esclusivamente su tweet in **inglese**: su testo in italiano tende a collassare su `neutral` indipendentemente dal contenuto. Ne teniamo conto nella Sezione 5, dove i post simulati per il monitoraggio sono scritti in inglese proprio per evitare questo effetto e ottenere una demo di drift detection realistica. In un contesto reale con post in italiano, la scelta corretta sarebbe un modello multilingue, ad es. [`cardiffnlp/twitter-xlm-roberta-base-sentiment`](https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment).


## 3. Fase 1 — Valutazione del modello sul dataset pubblico `tweet_eval`

Valutiamo l'accuratezza del modello pre-addestrato su un campione del validation set di `tweet_eval` (subset `sentiment`), per avere una baseline quantitativa delle performance prima del deploy in produzione.


In [ ]:
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

MODEL_LABELS = ["negative", "neutral", "positive"]  # ordine coerente col dataset tweet_eval

# NB: "tweet_eval" senza namespace non è più caricabile (HF ha rimosso il
# loading script legacy); il dataset ora vive sotto il namespace cardiffnlp.
ds = load_dataset("cardiffnlp/tweet_eval", "sentiment")["validation"]
N_SAMPLES = 300  # campione ridotto per velocità in demo; aumentare per una valutazione più robusta
ds_sample = ds.select(range(min(N_SAMPLES, len(ds))))

pred_raw = sentiment_pipe(list(ds_sample["text"]), truncation=True, batch_size=32)
y_pred = [MODEL_LABELS.index(r["label"].lower()) for r in pred_raw]
y_true = list(ds_sample["label"])

acc = accuracy_score(y_true, y_pred)
print(f"Accuracy su {len(ds_sample)} esempi: {acc:.4f}\n")
print(classification_report(y_true, y_pred, target_names=MODEL_LABELS))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(MODEL_LABELS)
ax.set_yticks(range(3)); ax.set_yticklabels(MODEL_LABELS)
ax.set_xlabel("Predetto"); ax.set_ylabel("Reale")
ax.set_title("Confusion Matrix — tweet_eval (validation sample)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max()/2 else "black")
fig.colorbar(im)
plt.tight_layout()
plt.show()


## 4. Fase 2 — Pipeline CI/CD

Il codice sorgente completo (script di training, valutazione, monitoraggio, test e app) è pubblicato nella repository GitHub linkata in cima al notebook, nella cartella `src/`, `tests/` e `.github/workflows/`.

**Design della pipeline** (`.github/workflows/ci-cd.yml`), eseguita automaticamente ad ogni push/PR su `main`:

1. **Test job** — lint (`flake8`) + unit/integration test (`pytest`) su `src/sentiment_model.py`, verificando che l'output rispetti lo schema atteso (etichetta ∈ {negative, neutral, positive}, score ∈ [0,1]).
2. **Build job** — verifica che tutte le dipendenze si installino correttamente e che il modulo di inferenza sia importabile.
3. **Deploy job (facoltativo)** — attivato solo sugli eventi di *release*, se è configurato il secret `HF_TOKEN`: pubblica l'app (`src/app.py`, interfaccia Gradio) su Hugging Face Spaces.

Questo garantisce che ogni modifica al codice sia automaticamente testata prima di raggiungere l'ambiente di produzione, riducendo il rischio di regressioni nel modello o nell'app.

Il file YAML completo della pipeline è disponibile in repository: `.github/workflows/ci-cd.yml`.


In [ ]:
# Anteprima del contenuto della pipeline CI/CD (per riferimento — file completo nella repo)
ci_cd_preview = """
name: CI/CD - Sentiment Analysis Pipeline
on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]
  release:
    types: [ published ]

jobs:
  test:      # lint + pytest
  build:     # verifica installazione dipendenze
  deploy:    # facoltativo, solo su release, se presente HF_TOKEN
"""
print(ci_cd_preview)


## 5. Fase 3 — Monitoraggio continuo e Drift Detection

Simuliamo l'arrivo nel tempo di batch di post social riguardanti l'azienda, e monitoriamo:
- la distribuzione del sentiment (% positivo/neutro/negativo) in ciascun batch;
- il **drift** rispetto a una baseline, calcolato come *total variation distance* tra le distribuzioni;
- il trigger automatico di **retraining** quando il drift supera una soglia (qui `0.15`).

Questo simula, ad esempio, l'insorgere improvviso di una crisi reputazionale (picco di sentiment negativo) che il sistema deve rilevare rapidamente.


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

DRIFT_THRESHOLD = 0.15

def compute_distribution(labels):
    n = len(labels)
    return {
        "negative": labels.count("negative") / n,
        "neutral": labels.count("neutral") / n,
        "positive": labels.count("positive") / n,
    }

def compute_drift(baseline, current):
    return 0.5 * sum(abs(baseline[k] - current[k]) for k in baseline)

# Batch simulati nel tempo: nel batch 3 avviene un picco improvviso di sentiment negativo.
# NB: i testi sono in inglese perché cardiffnlp/twitter-roberta-base-sentiment-latest
# è addestrato su tweet in inglese; con testi in italiano il modello collassa quasi
# sempre su "neutral" e il drift non verrebbe rilevato (per l'italiano andrebbe usato
# un modello multilingue, es. cardiffnlp/twitter-xlm-roberta-base-sentiment).
demo_batches = [
    ["I love this product!", "Great customer service", "Delivery was on time as always",
     "Nothing special but it's fine", "Average product, does the job"],
    ["Support staff was very kind", "Good build quality", "App is a bit slow but useful",
     "Fair price for what you get", "Would recommend it to a friend"],
    ["Terrible service, waited for hours", "Product arrived broken", "Never buying from here again",
     "Customer support is nonexistent", "Total disappointment, do not recommend"],
    ["Terrible service again", "Still having shipping problems", "App keeps crashing constantly",
     "Refund never arrived", "Overall a really negative experience"],
]

rows = []
baseline = None
now = datetime.utcnow()

for i, batch in enumerate(demo_batches):
    preds = sentiment_pipe(batch)
    labels = [p["label"].lower() for p in preds]
    dist = compute_distribution(labels)

    if baseline is None:
        baseline = dist
        drift = 0.0
    else:
        drift = compute_drift(baseline, dist)

    retrain = drift > DRIFT_THRESHOLD
    rows.append({
        "timestamp": (now + timedelta(hours=i)).isoformat(),
        "batch": i + 1,
        "pct_negative": round(dist["negative"], 3),
        "pct_neutral": round(dist["neutral"], 3),
        "pct_positive": round(dist["positive"], 3),
        "drift_score": round(drift, 3),
        "retrain_triggered": retrain,
    })

    if retrain:
        baseline = dist  # dopo il retraining la nuova distribuzione diventa la baseline

monitoring_df = pd.DataFrame(rows)
monitoring_df


In [ ]:
# Visualizzazione dell'andamento del sentiment nel tempo
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(monitoring_df["batch"], monitoring_df["pct_positive"], marker="o", label="Positivo", color="green")
ax.plot(monitoring_df["batch"], monitoring_df["pct_neutral"], marker="o", label="Neutro", color="gray")
ax.plot(monitoring_df["batch"], monitoring_df["pct_negative"], marker="o", label="Negativo", color="red")
for _, row in monitoring_df.iterrows():
    if row["retrain_triggered"]:
        ax.axvline(row["batch"], color="orange", linestyle="--", alpha=0.6)
ax.set_xlabel("Batch temporale")
ax.set_ylabel("% del batch")
ax.set_title("Andamento del sentiment nel tempo (linee tratteggiate = retraining triggerato)")
ax.legend()
plt.tight_layout()
plt.show()


## 6. Fase 3 — Retraining automatico

Quando il drift supera la soglia, il sistema di monitoraggio (`src/monitor.py` nella repository) invoca lo script di retraining (`src/train.py`), che effettua il fine-tuning del modello:

- sui nuovi dati etichettati raccolti (es. tramite feedback umano su post ambigui), oppure
- su un refresh del dataset pubblico `tweet_eval`, per mantenere aggiornata la capacità del modello di generalizzare.

Lo script salva le metriche (`accuracy`, `f1_macro`) prima e dopo il retraining, per verificare che il nuovo modello non peggiori le performance (fondamentale prima di promuoverlo in produzione).

`trigger_retraining()` in `src/monitor.py` esegue realmente questo comando via `subprocess.run` (non è più solo un placeholder che stampa due righe): quando la funzione è invocata con `dry_run=False` lancia il training vero e proprio, mentre nella demo locale di questo notebook e nella `__main__` di `monitor.py` viene usato `dry_run=True` per mostrare il comando senza eseguirlo, dato che non è presente un file `data/new_labeled_data.csv` reale né una GPU disponibile qui.

```bash
# Comando eseguito realmente dal sistema di monitoraggio (dry_run=False) quando il drift supera la soglia
python -m src.train --data_path ./data/new_labeled_data.csv --epochs 1
```

> Il training completo (fine-tuning su GPU) non viene eseguito in questo notebook per limiti di tempo/risorse della demo; il codice completo, testato, è disponibile in `src/train.py` nella repository, ed è collegato realmente (non più commentato) al sistema di monitoraggio in `src/monitor.py`.


## 7. Fase 3 — Deploy facoltativo su Hugging Face

Il modello e una semplice interfaccia (`src/app.py`, basata su Gradio) possono essere pubblicati come **Hugging Face Space**, per esporre un endpoint pubblico di inferenza integrabile in altri sistemi aziendali (es. dashboard di reputation management).

Il deploy è automatizzato nel job `deploy` della pipeline CI/CD, attivato sugli eventi di release, a condizione che sia configurato il secret `HF_TOKEN` nel repository GitHub.


In [ ]:
# Anteprima semplificata dell'app Gradio (codice completo in src/app.py)
app_preview = """
import gradio as gr
from src.sentiment_model import SentimentAnalyzer

analyzer = SentimentAnalyzer()

def classify(text):
    result = analyzer.predict([text])[0]
    return {result.label: result.score}

demo = gr.Interface(fn=classify, inputs="text", outputs="label",
                     title="MachineInnovators — Social Media Sentiment Monitor")
demo.launch()
"""
print(app_preview)


## 8. Conclusioni e risultati

- Il modello pre-addestrato `cardiffnlp/twitter-roberta-base-sentiment-latest` raggiunge un'accuracy solida sul dataset pubblico `tweet_eval` (vedi metriche in Sezione 3), confermandosi adatto all'uso su testo da social media senza necessità di training iniziale da zero.
- La pipeline CI/CD automatizza test, build e (facoltativamente) deploy, garantendo qualità e ripetibilità ad ogni modifica del codice.
- Il sistema di monitoraggio rileva correttamente variazioni improvvise nella distribuzione del sentiment (drift), come dimostrato nella simulazione in Sezione 5, e attiva automaticamente il retraining quando necessario.
- Il ciclo MLOps completo (inferenza → monitoraggio → drift detection → retraining → nuovo deploy) è così chiuso e automatizzato, riducendo l'intervento manuale richiesto per la gestione della reputazione online di MachineInnovators Inc.

**Possibili estensioni future:**
- Integrazione diretta con le API dei social network per l'ingestione real-time dei post;
- Dashboard di visualizzazione (es. Grafana/Streamlit) collegata al log di monitoraggio;
- A/B testing tra il modello in produzione e il modello ri-addestrato prima della promozione automatica.


## 9. Istruzioni per pubblicare la repository su GitHub

Il codice sorgente completo di questo progetto (identico a quello descritto/richiamato in questo notebook) è organizzato pronto per essere pubblicato:

```
repo/
├── src/            # sentiment_model.py, train.py, evaluate.py, monitor.py, app.py
├── tests/          # test_sentiment_model.py
├── .github/workflows/ci-cd.yml
├── data/README.md
├── requirements.txt
└── README.md       # documentazione completa del progetto
```

**Passi per pubblicarla:**

```bash
cd repo
git init
git add .
git commit -m "Initial commit: MachineInnovators sentiment monitoring project"
git branch -M main
git remote add origin https://github.com/<tuo-username>/<nome-repo>.git
git push -u origin main
```

Dopo il push, aggiorna il link in cima a questo notebook con l'URL reale della repository, poi ripubblica/condividi il notebook Colab come consegna finale del progetto.
